In [1]:
from sympy import *
from sympy.physics.quantum import TensorProduct as TP
import itertools as it

In [2]:
#we first define symbols for the elements of S_3 and for the equality patterns
E, T1, T2, T3, C1, C2, Eq, D = symbols('E, T_1, T_2, T_3, C_1, C_2, Eq, D')
#and then the set of permutations S_3, that is, the group
S3 = (E, T1, T2, T3, C1, C2)
Pi= (D, T1, T2, T3, Eq)     # see eq. (3.3) in the article
# Let us define the representation gamma - how the group S_3 acts on 
# 5-dim module R[\Pi] = span{ distinct | 12 eq, 23 eq, 31 eq | all equal} - see eq. (5.6)
e5 = eye(5)
gamma = { E: e5,
        T1: Matrix([e5[0,:],  #[1,0,0,0,0]
                    e5[1,:],  #[0,1,0,0,0]
                    e5[3,:],  #[0,0,0,1,0]
                    e5[2,:],  #[0,0,1,0,0]
                    e5[4,:]]),#[0,0,0,0,1]
       #
        T2: Matrix([e5[0,:],  #[1,0,0,0,0]
                    e5[3,:],  #[0,0,0,1,0]
                    e5[2,:],  #[0,0,1,0,0]
                    e5[1,:],  #[0,1,0,0,0]
                    e5[4,:]]),#[0,0,0,0,1]
       #
        T3: Matrix([e5[0,:],  #[1,0,0,0,0]
                    e5[2,:],  #[0,0,1,0,0]
                    e5[1,:],  #[0,1,0,0,0]
                    e5[3,:],  #[0,0,0,1,0]
                    e5[4,:]]),#[0,0,0,0,1]
       #
        C1: Matrix([e5[0,:],  #[1,0,0,0,0]
                    e5[3,:],  #[0,0,0,1,0]
                    e5[1,:],  #[0,1,0,0,0]
                    e5[2,:],  #[0,0,1,0,0]
                    e5[4,:]]),#[0,0,0,0,1]
       #
        C2: Matrix([e5[0,:],  #[1,0,0,0,0]
                    e5[2,:],  #[0,0,1,0,0]
                    e5[3,:],  #[0,0,0,1,0]
                    e5[1,:],  #[0,1,0,0,0]
                    e5[4,:]]),#[0,0,0,0,1]
}

In [3]:
# The module R[\Pi]^{S_3} decomposes as V \oplus W where V = R[\Pi]^{S_3}
# is the S3-invariant subspace
# The Reynolds projector \frac 1{\#G} \sum_{g \in G} gamma(g) onto V is:
PV = Rational(1,6) * sum(gamma.values(),zeros(5,5))
# A projector onto subspace W - orthogonal complement of V
PW = eye(5) - PV

e,t,c,s = symbols('e,t,c,s') # see eq. (3.8 -- 3.10) in the article. Additional symbol: s = e+3*t+2*c
#for each party we write down the matrix mapping from the space R[S_3] to R[\Pi]
S = Matrix([[1,0,0,0,0,0],
            [1,1,0,0,0,0],
            [1,0,1,0,0,0],
            [1,0,0,1,0,0],
            [1,1,1,1,1,1]])
#and the corresponding Weingarten matrix, W(d) in the article, eq. (3.11)
Wd = Matrix([[e,t,t,t,c,c],
            [t,e,c,c,t,t],
            [t,c,e,c,t,t],
            [t,c,c,e,t,t],
            [c,t,t,t,e,c],
            [c,t,t,t,c,e]])

#for a single subsystem, the coefficient matrix is defined as
Md = S*Wd[:,:-1] # last column removed
Md = Md.subs(e+3*t+2*c,s)

In [4]:
# We proceed to an explicit construction of the n-partite problem by setting the symbolic 
# vectors of invariants and equality patterns
# here and in the following lines, n will be used to denote the number of subsystems

def x_full(n):
    # eq. (5.1) in the article
    return Matrix([Symbol(f"x_{{{','.join(map(str, indices))}}}") for indices in it.product(S3, repeat=n)])

# but taking into account the fact that not all of them are accessible we proceed to the reduction
# according to which cycles orientations are not distinguishble 
#(in the article we use C (3.13) for the map from R^6 to R^5, here we call it Cmap)
e6=eye(6)

Cmap = Matrix([e6[0,:],        #[1,0,0,0,0,0]
               e6[1,:],        #[0,1,0,0,0,0]
               e6[2,:],        #[0,0,1,0,0,0]
               e6[3,:],        #[0,0,0,1,0,0]
               e6[4,:]+e6[5,:] #[0,0,0,0,1,1]
          ])

def Cn(n):
    #   C ⊗ C ⊗ ... ⊗ C (n times), a 5**n x 6**n matrix
    return TP(*[Cmap] * n)


#the available invariants are listed in the following vector
def z(n):
    # eq. (5.4)
    z = Cn(n)*x_full(n)
    return z

#Analogously we symbolically construct the vectors of all admissible values of correlation function
def y_patt(n):
    return Matrix([Symbol(f"y_{{{','.join(map(str, indices))}}}") for indices in it.product(Pi, repeat=n)])

#Then we build the coefficient matrix of the system y_patt= M(n)*z constructed party by party as follows
def Md_i(i):
    """returns the Md matrix with all entries indexed by a symbol passed in the argument i"""
    return Md.subs({Symbol(name): Symbol(f'{name}_{i}') for name in 'etcs'})

def M(n):
    """SW_i(1) ⊗ SW_i(2) ⊗ ... ⊗ SW_i(n)""" # eq. (5.5) in the article
    return TP(*[Md_i(i) for i in range(1, n + 1)])

In [13]:
# We show now how to construct the orbit space analysis and simplify the calculations exposed above

#First, we need to construct the matrix R_n^{sec} which is constructed starting from the projectors 
#PU and PW referred to monopartite analysis made above

def R_sec(n):
    cols = []
    # we construct the matrix whose columns are elements of bases of subsequent sectors H_X  (6.5) where
    # $X \in \{V, W\}^{\times n}$
    # below we iterate over all sectors, first iterating over k (number of W factors in X)
    # and then over all possible rearrangments of k W factors into n positions in X
    for k in range(n + 1):
        for Wpos in it.combinations(range(n), k): # Wpos stores positions of W factors in X
            P = sum(
                (
                    # here we take tensor products of gamma[g] restricted to factors V or W in H_X
                    TP(*[  PW*gamma[g]*PW if i in Wpos else PV  for i in range(n)  ])
                    for g in S3
                ),
                zeros(5**n, 5**n)
            ) / 6
            # and then we take the average over the group to obtain a Reynolds projector
            # onto the invariant subspace of a sector
            for v in P.columnspace():
                norm = sum(i != 0 for i in v)
                cols.append(norm * v) 
                # we append to the result the basis of the sector 
                # - linearly independent columns of P provided by P.columnspace
                # we revert the normalisation of basis vectors in agreement with 
                # the orbit space normalisation as in the article

    return Matrix.hstack(*cols)

#One can take the invariant projector and 
#gain easily its linearly independent columns that are NOT rearranged in such a way that they are organized with 
#respect to the decompostion R[\Pi]^{\otimes n}=(U \oplus W)^{\otimes n} but they are organized w.r. to orbit construction

def P(n): # Reynolds projector onto the invariant subspace of n-parties equality-pattern space
    P = sum((TP(*[gamma[g]] * n) for g in S3), zeros(5**n, 5**n)) / 6
    return P

# for a vector of equality patterns \pi in its orbit O(\pi),a respective basis vector e_\pi 
# P: e_\pi \mapsto \frac 1{|O(\pi)|} \sum_{\sigma \in O(\pi)} e_\sigma
# There are |O(\pi)| such identical columns in P
# P.columnspace collects subsequent columns related to \pi ordered lexicographicaly
# w.r.t. the basis order (D, T1, T2, T3, Eq) of Pi,
# orbit basis element appears in P.columnspace, when its first representant appears

def R_orb(n):
    Rcols = []
    for v in P(n).columnspace():           
        for entry in v:
            if entry:
                orbit_size = 1 / entry
                break
        Rcols.append(orbit_size * v) # we revert the orbit space normalisation as in the article
    return Matrix.hstack(*Rcols)

# Vectors appearing as columns of R_orb are not normalized but are orthogonal and this is the reason why in the formula of L_orb we can 
# assume that (R_orb.T * R_orb) is diagonal

def L_orb(n):
    return Matrix.hstack(*P(n).columnspace()).T 
    # here the normalisation is correct - the averaging factor, see article

# Here we perform the reduction by multiplying by matrices L and R, obtatining the square matrix acting inside the orbit space (which is not the tensor product anymore)
def M_orb(n):
    return L_orb(n) * M(n) * R_orb(n) # y_patt = M_orb z_orb

# then whe consider the transition matrix to go from sec analysis to orbit analysis
def Q(n):
    return L_orb(n) * R_sec(n)

# Finally we see how coefficient matrix in orbit space is with respect to the above transformation (denoted as Q in the article)
def M_sec(n):
    Qn=Q(n)
    Matrix_sec = Qn.inv() * M_orb(n) * Qn # y_sec = M z
    
    subs_dict = {Symbol(f'e_{i}') + 3*Symbol(f't_{i}') + 2*Symbol(f'c_{i}'): Symbol(f's_{i}')
        for i in range(1, n + 1)}
    
    return Matrix_sec.applyfunc(factor).subs(subs_dict)

In [14]:
#TESTS

In [15]:
# n = 2
M_orb(2)

Matrix([
[                e_1*e_2,                                           3*e_1*t_2,                 c_2*e_1,                                           3*e_2*t_1,                                                                   3*t_1*t_2,                                                                                               6*t_1*t_2,                                           3*c_2*t_1,                 c_1*e_2,                                           3*c_1*t_2,                 c_1*c_2],
[        e_1*(e_2 + t_2),                 2*e_1*(c_2 + t_2) + e_1*(e_2 + t_2),         e_1*(c_2 + t_2),                                   3*t_1*(e_2 + t_2),                                         2*t_1*(c_2 + t_2) + t_1*(e_2 + t_2),                                                                   4*t_1*(c_2 + t_2) + 2*t_1*(e_2 + t_2),                                   3*t_1*(c_2 + t_2),         c_1*(e_2 + t_2),                 2*c_1*(c_2 + t_2) + c_1*(e_2 + t_2),         c_1*(c_2 + t_2)],

In [16]:
M_sec(2)

Matrix([
[                e_1*e_2,         3*e_1*t_2,                 c_2*e_1,         3*e_2*t_1, 9*t_1*t_2,         3*c_2*t_1,                 c_1*e_2,         3*c_1*t_2,                 c_1*c_2,                       0],
[        e_1*(e_2 + t_2),           e_1*s_2,         e_1*(c_2 + t_2), 3*t_1*(e_2 + t_2), 3*s_2*t_1, 3*t_1*(c_2 + t_2),         c_1*(e_2 + t_2),           c_1*s_2,         c_1*(c_2 + t_2),                       0],
[                e_1*s_2,         3*e_1*s_2,                 e_1*s_2,         3*s_2*t_1, 9*s_2*t_1,         3*s_2*t_1,                 c_1*s_2,         3*c_1*s_2,                 c_1*s_2,                       0],
[        e_2*(e_1 + t_1), 3*t_2*(e_1 + t_1),         c_2*(e_1 + t_1),           e_2*s_1, 3*s_1*t_2,           c_2*s_1,         e_2*(c_1 + t_1), 3*t_2*(c_1 + t_1),         c_2*(c_1 + t_1),                       0],
[(e_1 + t_1)*(e_2 + t_2),   s_2*(e_1 + t_1), (c_2 + t_2)*(e_1 + t_1),   s_1*(e_2 + t_2),   s_1*s_2,   s_1*(c_2 + t_2), (c_1 + t_1)*(e_2

In [17]:
Q(2)

Matrix([
[1, 0, 0, 0, 0, 0, 0, 0, 0,  0],
[0, 1, 0, 0, 0, 0, 0, 0, 0,  0],
[0, 0, 1, 0, 0, 0, 0, 0, 0,  0],
[0, 0, 0, 1, 0, 0, 0, 0, 0,  0],
[0, 0, 0, 0, 1, 0, 0, 0, 0,  2],
[0, 0, 0, 0, 1, 0, 0, 0, 0, -1],
[0, 0, 0, 0, 0, 1, 0, 0, 0,  0],
[0, 0, 0, 0, 0, 0, 1, 0, 0,  0],
[0, 0, 0, 0, 0, 0, 0, 1, 0,  0],
[0, 0, 0, 0, 0, 0, 0, 0, 1,  0]])

In [18]:
# n = 3
M_orb(3)

Matrix([
[                        e_1*e_2*e_3,                                                               3*e_1*e_2*t_3,                         c_3*e_1*e_2,                                                               3*e_1*e_3*t_2,                                                                                                   3*e_1*t_2*t_3,                                                                                                                                           6*e_1*t_2*t_3,                                                               3*c_3*e_1*t_2,                         c_2*e_1*e_3,                                                               3*c_2*e_1*t_3,                         c_2*c_3*e_1,                                                               3*e_2*e_3*t_1,                                                                                                   3*e_2*t_1*t_3,                                                                               

In [19]:
M_sec(3)

Matrix([
[                        e_1*e_2*e_3,                 3*e_1*e_2*t_3,                         c_3*e_1*e_2,                 3*e_1*e_3*t_2,         9*e_1*t_2*t_3,                 3*c_3*e_1*t_2,                         c_2*e_1*e_3,                 3*c_2*e_1*t_3,                         c_2*c_3*e_1,                 3*e_2*e_3*t_1,         9*e_2*t_1*t_3,                 3*c_3*e_2*t_1,         9*e_3*t_1*t_2, 27*t_1*t_2*t_3,         9*c_3*t_1*t_2,                 3*c_2*e_3*t_1,         9*c_2*t_1*t_3,                 3*c_2*c_3*t_1,                         c_1*e_2*e_3,                 3*c_1*e_2*t_3,                         c_1*c_3*e_2,                 3*c_1*e_3*t_2,         9*c_1*t_2*t_3,                 3*c_1*c_3*t_2,                         c_1*c_2*e_3,                 3*c_1*c_2*t_3,                         c_1*c_2*c_3,                                   0,                             0,                                   0,                                   0,                           

In [20]:
Q(3)

Matrix([
[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,  0,  0,  0,  0,  0,  0,  2,  0,  0,  0],
[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,  0,  0,  0,  0,  0,  0, -1,  0,  0,  0],
[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
[0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
[0, 0, 